# 10. 딥러닝 - 다중분류 (Keras)

> 이진분류(09번 파일)와 출력층/손실함수가 달라 별도 파일로 분리했습니다. Wine 데이터셋(품종 3종)을 딥러닝으로 분류합니다.

### 이 노트북에서 배우는 것
- 이진분류 DL과 다중분류 DL 차이 다시 짚기
- 출력층 노드=클래스 수 + softmax 설계
- categorical_crossentropy vs sparse_categorical_crossentropy 차이
- softmax 출력을 실제 클래스 라벨로 바꾸는 np.argmax 후처리
- loss curve, 다중클래스 confusion matrix로 성능 확인

### 사용 데이터
- `data/wine.csv` (target 0/1/2 3개 클래스)

## 목차
1. 다중분류 DL이란? 이진분류 DL과 무엇이 다른가
2. 다중분류용 Sequential 모델 설계
3. 모델 컴파일
4. 모델 학습
5. 예측 결과 후처리
6. 성능 시각화
7. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(100)

wine = pd.read_csv('../data/wine.csv')
X = wine.drop(columns=['target'])
y = wine['target']
class_num = y.nunique()
print('클래스 수:', class_num)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train.shape

## 1. 다중분류 DL이란? 이진분류 DL과 무엇이 다른가

### 📖 이론 설명
09번 노트북(이진분류)과 비교하면 세 가지가 달라집니다.

1. **출력층 노드 수 = 클래스 수** (여기서는 3개). 하드코딩하지 않고 `y.nunique()`처럼 데이터에서 동적으로 계산해두면 다른 문제에도 코드를 재사용하기 좋습니다.
2. **activation='softmax'**: sigmoid는 '양성일 확률' 하나만 내지만, softmax는 **모든 클래스에 대한 확률을 동시에 내고, 그 합이 항상 1이 되도록** 만들어 줍니다.
3. **손실함수 선택**: 레이블 `y`가 정수(0,1,2)로 되어 있으면 `sparse_categorical_crossentropy`, 원-핫 인코딩된 벡터([1,0,0] 형태)면 `categorical_crossentropy`를 씁니다. 둘 다 개념은 같고 레이블의 '형태'만 다릅니다.

## 2. 다중분류용 Sequential 모델 설계

### 📖 이론 설명
출력층 노드 수는 `class_num`(=3)으로 지정하고 activation은 'softmax'로 고정합니다. 09번 노트북에서 다룬 `BatchNormalization`(학습 안정화)과 `Dropout`(과적합 방지)은 이진분류든 다중분류든 동일하게 `Dense` 층 사이에 넣어 조합할 수 있습니다.

### 🧩 핵심 개념 정리
| 레이어 | 노드 수 | activation |
|---|---|---|
| 은닉층 | 자유 | 'relu' |
| BatchNormalization | - | 층 출력값 재정규화 |
| 출력층(다중분류) | 클래스 수 | 'softmax' |

### 💻 예제 코드

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.2))
model.add(Dense(8, activation='relu'))
model.add(Dense(class_num, activation='softmax'))
model.summary()

### ✍️ TODO: 실전 문제

**문제 1.** 은닉층 32(relu)→16(relu), 출력층 `class_num`개(softmax)로 구성된 `model2`를 만들어 `summary()`를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model2 = Sequential()
model2.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model2.add(Dense(16, activation='relu'))
model2.add(Dense(class_num, activation='softmax'))  # 다중분류 출력층: 노드 수=클래스 수, softmax로 클래스별 확률 합이 1이 되도록 함
model2.summary()
```

</details>

## 3. 모델 컴파일

### 📖 이론 설명
`wine['target']`은 0,1,2 정수 레이블이므로(원-핫 인코딩되지 않은 상태) `sparse_categorical_crossentropy`를 사용합니다. 만약 `y`를 `to_categorical()`로 원-핫 인코딩했다면 `categorical_crossentropy`를 써야 합니다.

### 🧩 핵심 개념 정리
| y의 형태 | loss |
|---|---|
| 정수 레이블(0,1,2) | sparse_categorical_crossentropy |
| 원-핫 벡터([1,0,0]) | categorical_crossentropy |

### 💻 예제 코드

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

### ✍️ TODO: 실전 문제

**문제 1.** `model2`를 동일한 옵션으로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # y가 정수 레이블(0,1,2)이므로 sparse_categorical_crossentropy 사용(원-핫이면 categorical_crossentropy)
```

</details>

## 4. 모델 학습

### 📖 이론 설명
이진분류와 동일하게 `EarlyStopping`을 사용해 학습시킵니다.

### 💻 예제 코드

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, batch_size=16,
                     validation_data=(X_test, y_test), callbacks=[es], verbose=0)
print('학습된 epoch 수:', len(history.history['loss']))

### ✍️ TODO: 실전 문제

**문제 1.** `model2`를 `epochs=100`, `batch_size=16`, `EarlyStopping(patience=10)`로 학습시키세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
es2 = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history2 = model2.fit(X_train, y_train, epochs=100, batch_size=16, validation_data=(X_test, y_test), callbacks=[es2], verbose=0)  # 이진분류(09번)와 동일하게 EarlyStopping으로 조기 종료
print(len(history2.history['loss']))
```

</details>

## 5. 예측 결과 후처리

### 📖 이론 설명
`model.predict(X_test)`는 각 샘플에 대해 **클래스별 확률로 이루어진 벡터**(예: `[0.1, 0.7, 0.2]`)를 반환합니다. 이 중 **가장 큰 확률을 가진 위치(인덱스)**가 곧 예측 클래스이므로, `np.argmax(pred, axis=1)`로 최종 클래스 라벨을 얻습니다. `axis=1`은 '각 행(샘플)에서' 최댓값의 위치를 찾으라는 뜻입니다 - 이진분류에서 0.5 임계값을 쓰는 것과 대응되는, 다중분류의 표준적인 후처리 방법입니다.

### 🧩 핵심 개념 정리
| 코드 | 설명 |
|---|---|
| model.predict(X) | shape=(샘플수, 클래스수) 확률 벡터 반환 |
| np.argmax(pred, axis=1) | 각 행에서 확률이 가장 큰 클래스 인덱스 추출 |

### 💻 예제 코드

In [ ]:
pred_proba = model.predict(X_test)
print('예측 확률 벡터 shape:', pred_proba.shape)
print('첫 샘플 확률:', pred_proba[0])

y_pred = np.argmax(pred_proba, axis=1)
print('예측 클래스 라벨:', y_pred[:10])

### ✍️ TODO: 실전 문제

**문제 1.** `model2`의 예측 확률에서 첫 5개 샘플의 최종 클래스 라벨을 `argmax`로 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
pred2_proba = model2.predict(X_test)  # 각 샘플마다 클래스별 확률 벡터(합계 1)가 반환됨
pred2_label = np.argmax(pred2_proba, axis=1)  # axis=1: 각 행(샘플)에서 확률이 가장 큰 클래스의 인덱스를 찾음
print(pred2_label[:5])
```

</details>

## 6. 성능 시각화

### 📖 이론 설명
loss curve와 다중클래스 confusion matrix(3x3)로 성능을 확인합니다.

### 💻 예제 코드

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curve'); plt.legend(); plt.show()

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('DL Multi-class Confusion Matrix')
plt.show()

print('accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

### ✍️ TODO: 실전 문제

**문제 1.** `model2`의 예측 결과로 confusion matrix를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
pred2_proba = model2.predict(X_test)
pred2_label = np.argmax(pred2_proba, axis=1)  # confusion_matrix는 클래스 라벨을 받으므로 argmax로 변환한 뒤 넘겨야 함
cm2 = confusion_matrix(y_test, pred2_label)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()
```

</details>

## 7. 종합 실전 문제

**딥러닝 다중분류 전체 흐름을 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** 입력층 64(relu)+Dropout(0.2)→32(relu)→출력 `class_num`(softmax) 구조의 `model_final`을 만들고 `sparse_categorical_crossentropy`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model_final = Sequential()
model_final.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model_final.add(Dropout(0.2))  # 다중분류에서도 이진분류와 동일하게 Dropout으로 과적합을 방지할 수 있음
model_final.add(Dense(32, activation='relu'))
model_final.add(Dense(class_num, activation='softmax'))
model_final.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_final.summary()
```

</details>

**문제 2.** `model_final`을 학습시킨 뒤 예측 결과를 `argmax`로 변환해 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
es_f = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_final.fit(X_train, y_train, epochs=100, batch_size=16, validation_data=(X_test, y_test), callbacks=[es_f], verbose=0)
pred_f = np.argmax(model_final.predict(X_test), axis=1)  # softmax 확률 벡터를 argmax로 최종 클래스 라벨로 변환
print(accuracy_score(y_test, pred_f))
```

</details>